# QLoRA Fine-tuning: Llama 3.1/3.2 (1B–3B) on Medical Reasoning

**Dataset:** `OpenMed/Medical-Reasoning-SFT-Mega`  
**Technique:** QLoRA (4-bit quantization + LoRA adapters)  
**Platform:** Google Colab — T4 Free Tier  
**Libraries:** `transformers`, `peft`, `trl`, `bitsandbytes`, `datasets`, `wandb`/`mlflow`, `ollama`/`vllm`

---

## Notebook Flow
1. Install dependencies
2. Configure model & training hyperparameters
3. Load model in 4-bit (QLoRA)
4. Load & preview dataset
5. Before-training inference (baseline)
6. Fine-tune with SFTTrainer
7. After-training inference (comparison)
8. Save & push adapter to HuggingFace Hub
9. (Optional) Run with Ollama or vLLM

## 0. Check GPU & Runtime

In [1]:
import subprocess

# Check GPU
print("=== GPU Info ===")
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — make sure Runtime > Change runtime type > T4 GPU is selected!')

# Check available disk space
print("\n=== Disk Space ===")
result2 = subprocess.run(['df', '-h', '/'], capture_output=True, text=True)
print(result2.stdout)

# Check RAM
print("\n=== RAM ===")
result3 = subprocess.run(['free', '-h'], capture_output=True, text=True)
print(result3.stdout)

=== GPU Info ===
Thu Mar 19 19:01:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------------------

## 1. Install Dependencies

> **Restart the runtime after this cell!** (`Runtime → Restart runtime`)

In [ ]:
import subprocess, sys, pathlib, re

# 1. Install bitsandbytes 
print('\nInstalling bitsandbytes 0.45.3...')
!pip install -q 'bitsandbytes==0.45.3' --no-cache-dir

# 2. Core fine-tuning stack 
print('\nInstalling fine-tuning stack...')
!pip install -q \
    'transformers==4.46.3' \
    'peft==0.13.2' \
    'trl==0.12.2' \
    'accelerate==1.1.1' \
    'datasets==3.1.0' \
    sentencepiece \
    protobuf \
    huggingface_hub

# 3. Optional experiment tracking 
!pip install -q wandb mlflow

# 4. Verify 
print('\n── Verification ──')

bnb_dir = pathlib.Path('/usr/local/lib/python3.12/dist-packages/bitsandbytes')
binaries = sorted(bnb_dir.glob('libbitsandbytes_cuda*.so'))

if not binaries:
    # fallback: search wider
    binaries = sorted(pathlib.Path('/usr/local/lib').rglob('libbitsandbytes_cuda*.so'))

print('bnb CUDA binaries found:')
for b in binaries:
    marker =  if 'cuda128' in b.name else '    '
    print(f'{marker} {b.name}')

cuda128_ok = any('cuda128' in b.name for b in binaries)
if cuda128_ok:
    print('\n libbitsandbytes_cuda128.so present — correct for Colab T4 (CUDA 12.8)')
else:
    print('\n cuda128 binary missing!')
    print('   Try: !pip install bitsandbytes --upgrade --no-cache-dir')

# Version check (subprocess to avoid stale import cache)
r = subprocess.run(
    [sys.executable, '-c', 'import bitsandbytes as bnb; print("bnb", bnb.__version__)'],
    capture_output=True, text=True
)
print(r.stdout.strip() or r.stderr.strip())

print()
print('NOTE: Any "Could not find cuda128.so" WARNING in the subprocess above')
print('      is a stale-cache artifact from the current session — safe to ignore.')
print('      After RESTART the correct .so loads cleanly.')

print('\n' + '='*58)
print('   Installation complete.')
print('   RESTART THE RUNTIME NOW — this is required.')
print('  Runtime → Restart runtime  (or press Ctrl+M then .)')
print('  After restart: run Section 0 → 2 → 2b → 4 onward.')
print('  Do NOT re-run this Section 1 cell after restart.')
print('='*58)

Removing old bitsandbytes (if any)...
Found existing installation: bitsandbytes 0.45.3
Uninstalling bitsandbytes-0.45.3:
  Successfully uninstalled bitsandbytes-0.45.3
Done.

Installing bitsandbytes 0.45.3...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 152.0 MB/s eta 0:00:00

Installing fine-tuning stack...

── Verification ──
bnb CUDA binaries found:
     libbitsandbytes_cuda117.so
     libbitsandbytes_cuda118.so
     libbitsandbytes_cuda120.so
     libbitsandbytes_cuda121.so
     libbitsandbytes_cuda122.so
     libbitsandbytes_cuda123.so
     libbitsandbytes_cuda124.so
     libbitsandbytes_cuda125.so
     libbitsandbytes_cuda126.so
  ✅ libbitsandbytes_cuda128.so

✅ libbitsandbytes_cuda128.so present — correct for Colab T4 (CUDA 12.8)
bnb 0.45.3

NOTE: Any "Could not find cuda128.so" WARNING in the subprocess above
      is a stale-cache artifact from the current session — safe to ignore.
      After RESTART the correct .so loads cleanly.

  ✅  Installation complete.
  ⚠️

## 2. Configuration — Edit This Cell First!

In [ ]:
import os

# Llama 3.2 1B 
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

# Llama 3.2 3B 
# MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

# Llama 3.1 8B 
# MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

#  DATASET
DATASET_ID     = "OpenMed/Medical-Reasoning-SFT-Mega"
SUBSET_SIZE    = 100    # Small subset for quick testing
USE_FULL_DATA  = False   # Set True to train on full dataset


#  OUTPUT
OUTPUT_DIR     = "./llama-medical-qlora"
HF_USERNAME    = "aakashaldankar"   # For pushing to Hub (optional)
ADAPTER_NAME   = "llama-medical-qlora-adapter"


#  QLoRA HYPERPARAMETERS
LORA_R         = 16      # LoRA rank — higher = more params, more expressiveness
LORA_ALPHA     = 32      # LoRA scaling factor (typically 2x rank)
LORA_DROPOUT   = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]  # All linear layers


#  TRAINING HYPERPARAMETERS
NUM_EPOCHS           = 2        
PER_DEVICE_BATCH     = 2        
GRAD_ACCUM_STEPS     = 8        
LEARNING_RATE        = 2e-4
MAX_SEQ_LENGTH       = 1024     
WARMUP_RATIO         = 0.05
LR_SCHEDULER         = "cosine"
LOGGING_STEPS        = 10
SAVE_STEPS           = 100


#  EXPERIMENT TRACKING (set one)
USE_WANDB   = True   # Set True + add your WANDB_API_KEY
USE_MLFLOW  = False   # Set True for local mlflow tracking


#  HUGGINGFACE AUTH 
HF_TOKEN = os.environ.get("HF_TOKEN", "")

print(" Configuration loaded.")
print(f" Model      : {MODEL_ID}")
print(f" Dataset    : {DATASET_ID}")
print(f" Subset size: {SUBSET_SIZE if not USE_FULL_DATA else 'FULL'}")
print(f" LoRA rank  : {LORA_R}")
print(f" Epochs     : {NUM_EPOCHS}")

✅ Configuration loaded.
   Model      : meta-llama/Llama-3.2-1B-Instruct
   Dataset    : OpenMed/Medical-Reasoning-SFT-Mega
   Subset size: 100
   LoRA rank  : 16
   Epochs     : 2


## 2b. Authenticate HuggingFace

Llama models are **gated**. You must:
1. Create a free HuggingFace account
2. Accept the Llama license at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
3. Create a read token at https://huggingface.co/settings/tokens
4. Add it as a Colab Secret named HF_TOKEN (key icon in the left panel)

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    # Try Colab Secrets first (recommended)
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print(" Logged in via Colab Secrets.")
except Exception:
    if HF_TOKEN:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print(" Logged in via config variable.")
    else:
        print(" No HF_TOKEN found. Using interactive login...")
        login()   # Will prompt for token

✅ Logged in via Colab Secrets.


## 3. Load Dataset

In [4]:
from datasets import load_dataset
import pandas as pd

print(f"Loading dataset: {DATASET_ID}")

# Load dataset (streaming=True avoids downloading all data upfront)
raw_dataset = load_dataset(DATASET_ID, split="train")

print(f"\nFull dataset size: {len(raw_dataset):,} samples")
print(f"Columns: {raw_dataset.column_names}")
print(f"\nSample entry:")
print(raw_dataset[0])

Loading dataset: OpenMed/Medical-Reasoning-SFT-Mega


Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/29 [00:00<?, ?it/s]


Full dataset size: 1,789,998 samples
Columns: ['messages']

Sample entry:
{'messages': [{'content': 'Which of the following is **not** a common method by which bacteria can acquire new genetic material?\nA: Conjugation\nB: Transduction\nC: Transformation\nD: Binary fission\nE: Plasmid uptake\nF: Integron shuffling\nG: Transposition\nH: Phage-mediated gene transfer\nI: Gene cassettes integration\nJ: Horizontal gene transfer via conjugative elements', 'reasoning_content': None, 'role': 'user'}, {'content': 'The question asks which option is not a common method for bacteria to acquire new genetic material. Bacteria acquire new genetic material primarily through horizontal gene transfer mechanisms, which involve the uptake or transfer of DNA from external sources. The options include:\n\n- **A: Conjugation**: A process where bacteria transfer genetic material directly through cell-to-cell contact, such as via a pilus. This is a common method.\n- **B: Transduction**: A process where bacter

In [5]:
from datasets import Dataset

# ── Subset selection ──────────────────────────────────────────────
if USE_FULL_DATA:
    dataset = raw_dataset
    print(f"Using FULL dataset: {len(dataset):,} samples")
else:
    dataset = raw_dataset.select(range(min(SUBSET_SIZE, len(raw_dataset))))
    print(f"Using SUBSET: {len(dataset):,} samples (set USE_FULL_DATA=True for full training)")

# ── Inspect schema ────────────────────────────────────────────────
# The OpenMed dataset typically has 'instruction', 'output' or 'messages' fields.
# Adapt the formatter below to match actual column names.
sample = dataset[0]
print("\nDataset columns:", list(sample.keys()))

# Train / eval split (90/10)
split = dataset.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]
eval_data  = split["test"]

print(f"\nTrain: {len(train_data):,} | Eval: {len(eval_data):,}")

Using SUBSET: 100 samples (set USE_FULL_DATA=True for full training)

Dataset columns: ['messages']

Train: 90 | Eval: 10


In [ ]:
# Inspect real column names 
sample = dataset[0]
print('Dataset columns:', list(sample.keys()))
print('\nFirst example (truncated):')
for k, v in sample.items():
    print(f'  {k}: {str(v)[:200]}')

# System prompt 
SYSTEM_PROMPT = (
    'You are a highly knowledgeable medical assistant. '
    'Answer the following clinical question accurately, step-by-step, '
    'citing reasoning and evidence where applicable.'
)

# Define formatter
def format_to_text(example, tok):
    """
    Converts a dataset row to a single rendered string using the
    tokenizer's chat template.  Works with both:
      • 'messages' schema  (list-of-dicts)
      • flat 'instruction'/'output' schema
    """
    cols = set(example.keys())

    if 'messages' in cols:
        messages = list(example['messages'])
        # Ensure system message is present
        if not messages or messages[0].get('role') != 'system':
            messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + messages
    else:
        instruction = (
            example.get('instruction') or example.get('input')
            or example.get('question')  or example.get('prompt') or ''
        )
        output = (
            example.get('output')    or example.get('response')
            or example.get('answer') or example.get('completion') or ''
        )
        messages = [
            {'role': 'system',    'content': SYSTEM_PROMPT},
            {'role': 'user',      'content': str(instruction).strip()},
            {'role': 'assistant', 'content': str(output).strip()},
        ]

    # add_generation_prompt=False → includes the assistant turn (training mode)
    text = tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

print('\n Formatter defined. Dataset will be formatted in Section 3b')
print('   (after the tokenizer is loaded in Section 4).')


Dataset columns: ['messages']

First example (truncated):
  messages: [{'content': 'Which of the following is **not** a common method by which bacteria can acquire new genetic material?\nA: Conjugation\nB: Transduction\nC: Transformation\nD: Binary fission\nE: Plasmid u

✅ Formatter defined. Dataset will be formatted in Section 3b
   (after the tokenizer is loaded in Section 4).


## 4. Load Base Model (4-bit QLoRA)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import pathlib

# Sanity check 1: CUDA 
assert torch.cuda.is_available(), (
    "CUDA not available! Go to Runtime → Change runtime type → T4 GPU, then restart."
)
print(f" CUDA {torch.version.cuda} | GPU: {torch.cuda.get_device_name(0)}")

# Sanity check 2: bitsandbytes CUDA binary
import bitsandbytes as bnb
bnb_dir = pathlib.Path(bnb.__file__).parent
cuda_so_files = sorted(bnb_dir.glob("libbitsandbytes_cuda*.so"))
if not cuda_so_files:
    raise RuntimeError(
        "No bitsandbytes CUDA binary found!\n"
        "Re-run Section 1 (install cell), then Runtime → Restart runtime."
    )
# Pick the .so that best matches the current CUDA version
cuda_ver = torch.version.cuda.replace(".", "")          # e.g. "128"
best_so  = next((s for s in cuda_so_files if cuda_ver in s.name), cuda_so_files[-1])
print(f"bitsandbytes {bnb.__version__} | using {best_so.name}")

# 4-bit quantization config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Tokenizer
print(f"\nLoading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side="right",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token     = tokenizer.eos_token
    tokenizer.pad_token_id  = tokenizer.eos_token_id

print(f"Vocab size    : {tokenizer.vocab_size:,}")
print(f"Pad token     : {tokenizer.pad_token}")
print(f"Chat template : {'Yes' if tokenizer.chat_template else 'No'}")

# Base model
print(f"\nLoading model: {MODEL_ID} in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
base_model.config.use_cache      = False
base_model.config.pretraining_tp = 1

total_params     = sum(p.numel() for p in base_model.parameters())
trainable_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
print(f"\n Model loaded.")
print(f"   Total params    : {total_params/1e9:.2f}B")
print(f"   Trainable params: {trainable_params/1e6:.2f}M (before LoRA)")


✅ CUDA 12.8 | GPU: Tesla T4
✅ bitsandbytes 0.45.3 | using libbitsandbytes_cuda128.so

Loading tokenizer: meta-llama/Llama-3.2-1B-Instruct
Vocab size    : 128,000
Pad token     : <|eot_id|>
Chat template : Yes

Loading model: meta-llama/Llama-3.2-1B-Instruct in 4-bit...

✅ Model loaded.
   Total params    : 0.75B
   Trainable params: 262.74M (before LoRA)


In [ ]:
# Section 3b — Apply chat template

print('Applying chat template to train / eval sets...')

train_data = train_data.map(
    lambda ex: format_to_text(ex, tokenizer),
    remove_columns=train_data.column_names,
    desc='Formatting train set',
)
eval_data = eval_data.map(
    lambda ex: format_to_text(ex, tokenizer),
    remove_columns=eval_data.column_names,
    desc='Formatting eval set',
)

print('\n Dataset formatted — column is now "text" (plain string).')
print(f'   Train: {len(train_data):,}  |  Eval: {len(eval_data):,}')
print('\nFormatted sample (first 600 chars):')
print(train_data[0]['text'][:600])


Applying chat template to train / eval sets...


Formatting train set:   0%|          | 0/90 [00:00<?, ? examples/s]

Formatting eval set:   0%|          | 0/10 [00:00<?, ? examples/s]


✅ Dataset formatted — column is now "text" (plain string).
   Train: 90  |  Eval: 10

Formatted sample (first 600 chars):
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 19 Mar 2026

You are a knowledgeable and compassionate medical assistant. Your role is to provide accurate, evidence-based medical information while being clear and helpful.

Guidelines:
- Provide thorough, well-reasoned answers based on current medical knowledge
- Explain medical concepts in clear, accessible language
- When appropriate, mention when someone should seek professional medical care
- Be empathetic and supportive in your responses
- If a question is outside your expertis


## 5. EFORE Training — Baseline Inference

We run the **untrained model** on a fixed set of medical questions. We'll compare these answers to the fine-tuned model later.

In [ ]:
# Benchmark questions: these will be tested before AND after training
BENCHMARK_QUESTIONS = [
    "What is the first-line treatment for Type 2 Diabetes Mellitus in a newly diagnosed patient?",
    "A 65-year-old male presents with sudden onset chest pain radiating to the left arm, diaphoresis, and nausea. What is the most likely diagnosis and immediate management?",
    "Explain the mechanism of action of beta-lactam antibiotics and describe two mechanisms of bacterial resistance.",
    "What are the diagnostic criteria for Systemic Lupus Erythematosus (SLE) according to the 2019 EULAR/ACR classification?",
    "A child presents with high fever, stiff neck, photophobia, and a petechial rash. What is the differential diagnosis and emergency treatment?",
]


def run_inference(model, tokenizer_obj, question, max_new_tokens=400, label="Model"):
    """Run a single inference pass and return the generated answer."""
    messages = [
        {"role": "system",  "content": SYSTEM_PROMPT},
        {"role": "user",    "content": question},
    ]
    # Format using the model's chat template
    prompt = tokenizer_obj.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer_obj(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer_obj.pad_token_id,
            eos_token_id=tokenizer_obj.eos_token_id,
        )

    # Decode only newly generated tokens
    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer_obj.decode(new_tokens, skip_special_tokens=True)
    return answer.strip()


def pretty_print_comparison(question, before, after=None):
    """Pretty-print a before/after comparison."""
    sep = "─" * 70
    print(f"\n{'═'*70}")
    print(f"QUESTION: {question}")
    print(sep)
    print(f"BEFORE FINE-TUNING:\n{before}")
    if after is not None:
        print(sep)
        print(f"AFTER FINE-TUNING:\n{after}")
    print(f"{'═'*70}\n")

print("✅ Inference helpers ready.")

✅ Inference helpers ready.


In [ ]:
# Run baseline inference
print(" Running BASELINE inference (before fine-tuning)...\n")

before_answers = {}
for i, q in enumerate(BENCHMARK_QUESTIONS, 1):
    print(f"[{i}/{len(BENCHMARK_QUESTIONS)}] Generating answer...")
    answer = run_inference(base_model, tokenizer, q)
    before_answers[q] = answer
    pretty_print_comparison(q, answer)

print("\n Baseline inference complete. Answers stored in `before_answers`.")

🔴 Running BASELINE inference (before fine-tuning)...

[1/5] Generating answer...

══════════════════════════════════════════════════════════════════════
❓ QUESTION: What is the first-line treatment for Type 2 Diabetes Mellitus in a newly diagnosed patient?
──────────────────────────────────────────────────────────────────────
🔴 BEFORE FINE-TUNING:
For a newly diagnosed patient with Type 2 Diabetes Mellitus (T2DM), the first-line treatment typically involves lifestyle modifications and/or oral medications. Here's a step-by-step approach:

**Lifestyle Modifications:**

1. **Weight Loss**: Aiming to lose 5-10% of body weight within 12 weeks can help improve insulin sensitivity and reduce HbA1c levels.
2. **Dietary Changes**: Focus on whole, unprocessed foods, including:
	* Vegetables (aim for 5 servings/day)
	* Fruits (aim for 4 servings/day)
	* Whole grains (aim for 6 servings/day)
	* Lean protein sources (aim for 2 servings/day)
	* Healthy fats (limit saturated fat intake)
3. **Physical

## 6. Apply QLoRA (LoRA Adapters) with PEFT

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Prepare model for k-bit training (required before adding LoRA)
base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=True,  # Saves memory at cost of speed
)

# LoRA configuration
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
)

model = get_peft_model(base_model, lora_config)

# Print trainable parameters summary
model.print_trainable_parameters()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
pct       = 100 * trainable / total

print(f"\n QLoRA applied.")
print(f"   Total params    : {total/1e6:.1f}M")
print(f"   Trainable (LoRA): {trainable/1e6:.3f}M ({pct:.2f}%)")
print(f"   Frozen (4-bit)  : {(total - trainable)/1e6:.1f}M")

trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039

✅ QLoRA applied.
   Total params    : 760.5M
   Trainable (LoRA): 11.272M (1.48%)
   Frozen (4-bit)  : 749.3M


## 7. Experiment Tracking (Optional — WandB / MLflow)

In [ ]:
import os

if USE_WANDB:
    import wandb
    from google.colab import userdata

    try:
        wandb_key = userdata.get('WANDB_API_KEY')
    except Exception:
        wandb_key = os.environ.get('WANDB_API_KEY', '')

    if wandb_key:
        wandb.login(key=wandb_key)
    else:
        wandb.login()   # Interactive

    run = wandb.init(
        project="llama-medical-qlora",
        name=f"qlora-r{LORA_R}-{MODEL_ID.split('/')[-1]}",
        config={
            "model": MODEL_ID,
            "dataset": DATASET_ID,
            "subset_size": len(train_data),
            "lora_r": LORA_R,
            "lora_alpha": LORA_ALPHA,
            "epochs": NUM_EPOCHS,
            "lr": LEARNING_RATE,
            "batch_size": PER_DEVICE_BATCH * GRAD_ACCUM_STEPS,
            "max_seq_length": MAX_SEQ_LENGTH,
        },
    )
    print(f" W&B run started: {run.name}")
    os.environ["WANDB_PROJECT"] = "llama-medical-qlora"
    report_to = "wandb"

elif USE_MLFLOW:
    import mlflow
    mlflow.set_experiment("llama-medical-qlora")
    mlflow.start_run(run_name=f"qlora-r{LORA_R}-{MODEL_ID.split('/')[-1]}")
    mlflow.log_params({
        "model": MODEL_ID, "lora_r": LORA_R, "lr": LEARNING_RATE,
        "epochs": NUM_EPOCHS, "subset_size": len(train_data),
    })
    print(" MLflow run started.")
    report_to = "mlflow"   # note: HF doesn't have native mlflow; logs will be manual

else:
    os.environ["WANDB_DISABLED"] = "true"
    report_to = "none"
    print("ℹ Experiment tracking disabled. Set USE_WANDB=True or USE_MLFLOW=True to enable.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: haldankar-aakash-01 (haldankar-aakash-01-proviz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ W&B run started: qlora-r16-Llama-3.2-1B-Instruct


## 8. Fine-tune with SFTTrainer

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

# Training arguments optimized for T4 16GB
training_args = SFTConfig(
    # Output 
    output_dir=OUTPUT_DIR,
    run_name=f"qlora-r{LORA_R}",

    # Schedule 
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    # Optimizer 
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    optim="paged_adamw_8bit",   # Memory-efficient optimizer for QLoRA

    # Memory
    max_seq_length=MAX_SEQ_LENGTH,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=not torch.cuda.is_bf16_supported(),   
    bf16=torch.cuda.is_bf16_supported(),
    dataloader_pin_memory=True,

    # Logging & Saving 
    logging_steps=LOGGING_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to=report_to,

    # Dataset
    dataset_text_field="text",       
    packing=False,                   

    seed=42,
    group_by_length=True, 
    push_to_hub=False,
)

print(" Training arguments configured.")
print(f"   Effective batch size: {PER_DEVICE_BATCH * GRAD_ACCUM_STEPS}")
print(f"   Precision: {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")

✅ Training arguments configured.
   Effective batch size: 16
   Precision: bf16


In [ ]:
# SFTTrainer setup 
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,   # trl 0.12+ uses processing_class, not tokenizer=
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=training_args,
)

print(' SFTTrainer ready.')
print(f'   Training steps : {len(trainer.get_train_dataloader()) * NUM_EPOCHS}')
print(f'   Eval samples   : {len(eval_data)}')


Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

✅ SFTTrainer ready.
   Training steps : 90
   Eval samples   : 10


In [ ]:
import time

# TRAIN 
print(" Starting fine-tuning...\n")
start_time = time.time()

train_result = trainer.train()

elapsed = time.time() - start_time
print(f"\n Training complete in {elapsed/60:.1f} minutes.")
print(f"   Final train loss : {train_result.training_loss:.4f}")

# Log metrics
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)

🚀 Starting fine-tuning...



Step,Training Loss,Validation Loss



✅ Training complete in 10.5 minutes.
   Final train loss : 2.0504
***** train metrics *****
  epoch                    =     1.8444
  total_flos               =   782520GF
  train_loss               =     2.0504
  train_runtime            = 0:10:26.20
  train_samples_per_second =      0.287
  train_steps_per_second   =      0.016


In [ ]:
# Save LoRA adapter 
adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f" LoRA adapter saved to: {adapter_path}")
!ls -lh {adapter_path}

✅ LoRA adapter saved to: ./llama-medical-qlora/final_adapter
total 60M
-rw-r--r-- 1 root root  735 Mar 19 19:37 adapter_config.json
-rw-r--r-- 1 root root  44M Mar 19 19:37 adapter_model.safetensors
-rw-r--r-- 1 root root 5.0K Mar 19 19:37 README.md
-rw-r--r-- 1 root root  325 Mar 19 19:37 special_tokens_map.json
-rw-r--r-- 1 root root  54K Mar 19 19:37 tokenizer_config.json
-rw-r--r-- 1 root root  17M Mar 19 19:37 tokenizer.json


## 9. AFTER Training — Fine-tuned Inference & Comparison

In [ ]:
from peft import PeftModel

# Enable inference cache
model.config.use_cache = True
model.eval()

print(" Running FINE-TUNED inference (after training)...\n")

after_answers = {}
for i, q in enumerate(BENCHMARK_QUESTIONS, 1):
    print(f"[{i}/{len(BENCHMARK_QUESTIONS)}] Generating answer...")
    answer = run_inference(model, tokenizer, q)
    after_answers[q] = answer

print("\n Fine-tuned inference complete.")

🟢 Running FINE-TUNED inference (after training)...

[1/5] Generating answer...
[2/5] Generating answer...
[3/5] Generating answer...
[4/5] Generating answer...
[5/5] Generating answer...

✅ Fine-tuned inference complete.


In [ ]:
# Side-by-side comparison 
print("\n" + "★" * 70)
print("         BEFORE vs AFTER FINE-TUNING — COMPARISON REPORT")
print("★" * 70)

for q in BENCHMARK_QUESTIONS:
    pretty_print_comparison(
        question=q,
        before=before_answers[q],
        after=after_answers[q],
    )


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
         BEFORE vs AFTER FINE-TUNING — COMPARISON REPORT
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

══════════════════════════════════════════════════════════════════════
❓ QUESTION: What is the first-line treatment for Type 2 Diabetes Mellitus in a newly diagnosed patient?
──────────────────────────────────────────────────────────────────────
🔴 BEFORE FINE-TUNING:
For a newly diagnosed patient with Type 2 Diabetes Mellitus (T2DM), the first-line treatment typically involves lifestyle modifications and/or oral medications. Here's a step-by-step approach:

**Lifestyle Modifications:**

1. **Weight Loss**: Aiming to lose 5-10% of body weight within 12 weeks can help improve insulin sensitivity and reduce HbA1c levels.
2. **Dietary Changes**: Focus on whole, unprocessed foods, including:
	* Vegetables (aim for 5 servings/day)
	* Fruits (aim for 4 servings/day)
	* Whole grains (aim for 6 se

In [ ]:
# DataFrame for easy review
import pandas as pd

rows = []
for q in BENCHMARK_QUESTIONS:
    rows.append({
        "Question"    : q[:80] + "...",
        "Before (len)": len(before_answers[q]),
        "After (len)" : len(after_answers[q]),
        "Before"      : before_answers[q][:300],
        "After"       : after_answers[q][:300],
    })

comparison_df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 200)
comparison_df[["Question", "Before (len)", "After (len)"]]

,Question,Before (len),After (len)
0,What is the first-line treatment for Type 2 Diabetes Mellitus in a newly diagnos...,1789,1538
1,A 65-year-old male presents with sudden onset chest pain radiating to the left a...,1841,1651
2,Explain the mechanism of action of beta-lactam antibiotics and describe two mech...,1858,1805
3,What are the diagnostic criteria for Systemic Lupus Erythematosus (SLE) accordin...,1464,1552
4,"A child presents with high fever, stiff neck, photophobia, and a petechial rash....",1823,1610


## 10. Merge LoRA Weights & Save Full Model

This merges the LoRA adapter back into the base model weights for standalone deployment (no PEFT dependency needed at inference time).

In [ ]:

MERGE_AND_SAVE = True  

if MERGE_AND_SAVE:
    from peft import AutoPeftModelForCausalLM

    print("Loading adapter from disk for merge...")
    merged_model = AutoPeftModelForCausalLM.from_pretrained(
        adapter_path,
        low_cpu_mem_usage=True,
        torch_dtype=torch.float16,
        device_map="auto",
    )

    print("Merging LoRA weights into base model...")
    merged_model = merged_model.merge_and_unload()

    merged_path = os.path.join(OUTPUT_DIR, "merged_model")
    merged_model.save_pretrained(merged_path, safe_serialization=True)
    tokenizer.save_pretrained(merged_path)

    print(f" Merged model saved to: {merged_path}")
    !ls -lh {merged_path}
else:
    print("ℹ Merge skipped. Using adapter-only model.")

Loading adapter from disk for merge...
Merging LoRA weights into base model...
✅ Merged model saved to: ./llama-medical-qlora/merged_model
total 2.4G
-rw-r--r-- 1 root root  926 Mar 19 19:42 config.json
-rw-r--r-- 1 root root  184 Mar 19 19:42 generation_config.json
-rw-r--r-- 1 root root 2.4G Mar 19 19:43 model.safetensors
-rw-r--r-- 1 root root  325 Mar 19 19:43 special_tokens_map.json
-rw-r--r-- 1 root root  54K Mar 19 19:43 tokenizer_config.json
-rw-r--r-- 1 root root  17M Mar 19 19:43 tokenizer.json


## 11. Push Adapter to HuggingFace Hub

In [ ]:
PUSH_TO_HUB = False   # Set True to push

if PUSH_TO_HUB and HF_USERNAME:
    repo_id = f"{HF_USERNAME}/{ADAPTER_NAME}"
    print(f"Pushing adapter to: {repo_id}")

    trainer.model.push_to_hub(repo_id, private=True)
    tokenizer.push_to_hub(repo_id, private=True)

    print(f" Adapter pushed to: https://huggingface.co/{repo_id}")
else:
    print("ℹHub push skipped. Set PUSH_TO_HUB=True and HF_USERNAME to enable.")

## 12. Run with Ollama

Ollama is a lightweight local inference server. It requires the **merged** model exported to GGUF format.

> **Note:** Ollama runs best **locally** (not on Colab), since it needs a persistent server process. The cells below show how to export and use it.

In [ ]:
USE_OLLAMA = True 

if USE_OLLAMA:
    import subprocess, time, requests, os, pathlib

    print("Installing zstd dependency...")
    !apt-get install -qq -y zstd

    print("\nInstalling Ollama...")
    !curl -fsSL https://ollama.com/install.sh | sh

    if not pathlib.Path("/usr/local/bin/ollama").exists():
        raise FileNotFoundError(
            "Ollama binary not found after install.\n"
            "Try running: !curl -fsSL https://ollama.com/install.sh | sh"
        )
    print(" Ollama installed:", subprocess.run(["ollama", "--version"],
          capture_output=True, text=True).stdout.strip())

    print("\nStarting Ollama server...")
    server_env = os.environ.copy()
    server_env["OLLAMA_HOST"] = "0.0.0.0:11434"

    server_proc = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        env=server_env,
    )

    for attempt in range(30):
        time.sleep(1)
        try:
            r = requests.get("http://localhost:11434", timeout=1)
            if r.status_code == 200:
                print(f" Ollama server ready (after {attempt+1}s).")
                break
        except Exception:
            pass
    else:
        # Print stderr to help debug
        err = server_proc.stderr.read(2000).decode(errors="replace")
        raise RuntimeError(f"Ollama server did not start in 30s.\nStderr: {err}")


    OLLAMA_MODEL = "llama3.2:1b"
    print(f"\nPulling {OLLAMA_MODEL} from Ollama Hub...")
    print("(This downloads ~2 GB — takes a few minutes on Colab)")
    !ollama pull {OLLAMA_MODEL}


    # Uncomment this block if you ran Section 10 (merge) and want to test
    # your fine-tuned model via Ollama instead of the base model.
    #
    # merged_path = os.path.abspath(os.path.join(OUTPUT_DIR, "merged_model"))
    # if os.path.isdir(merged_path):
    #     print(f"\nCreating Ollama model from merged weights: {merged_path}")
    #     modelfile = f'''FROM {merged_path}
    # SYSTEM "{SYSTEM_PROMPT}"
    # PARAMETER temperature 0.3
    # PARAMETER top_p 0.9
    # '''
    #     with open("/tmp/Modelfile", "w") as f:
    #         f.write(modelfile)
    #     !ollama create llama-medical -f /tmp/Modelfile
    #     OLLAMA_MODEL = "llama-medical"
    # else:
    #     print(f" Merged model not found at {merged_path}. Using pulled model.")

    print(f"\n Running inference with Ollama ({OLLAMA_MODEL})...")

    for i, question in enumerate(BENCHMARK_QUESTIONS, 1):
        print(f"\n[{i}/{len(BENCHMARK_QUESTIONS)}] ❓ {question[:80]}...")
        try:
            resp = requests.post(
                "http://localhost:11434/api/generate",
                json={
                    "model": OLLAMA_MODEL,
                    "prompt": question,
                    "system": SYSTEM_PROMPT,
                    "stream": False,
                    "options": {"temperature": 0.3, "top_p": 0.9},
                },
                timeout=120,
            )
            resp.raise_for_status()
            answer = resp.json().get("response", "(no response)")
            print(f" {answer[:400]}")
        except Exception as e:
            print(f" Error: {e}")

    print("\n Ollama inference complete.")

else:
    print("ℹ Ollama skipped. Set USE_OLLAMA=True to enable.")


Installing zstd dependency...
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...

Installing Ollama...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
✅ Ollama installed: Warning: could not connect to a running Ollama instance

Starting Ollama server...
✅ Ollama server ready (after 2s).

Pulling llama3.2:3b from 